# SAGE-3D 场景面积分布分析

从 `SAGE-3D_Official/SAGE-3D_data/semantic_maps/` 下的 JSON 文件中，
通过 `mask_coords_m` 坐标范围计算每个场景的**可通行面积（m²）**，并给出行人数量推荐。

### 面积计算策略
每个 JSON 里每个实例都有 `area`（像素数）和 `mask_coords_m`（真实坐标列表）。
- **方案A（推荐）**：用所有实例的 `mask_coords_m` 推算场景坐标边界 → 总面积；
  再单独提取 `floor` 类的像素坐标做并集 → 地板面积（即可通行面积估计）。
- **方案B（简单）**：直接把 `floor` 类所有实例的 `area`（像素数）求和，
  再乘以 `scale²`（0.05² = 0.0025 m²/pixel）换算成 m²。

本 Notebook 同时提供两种方案，方便对比。

In [ ]:
import json
import glob
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import rcParams

rcParams['font.family'] = 'DejaVu Sans'
rcParams['axes.titlesize'] = 13
rcParams['axes.labelsize'] = 11

# ── 路径配置 ──────────────────────────────────────────────────────────────────
MAP_DIR = Path("/workspace/SAGE-3D_Official/SAGE-3D_data/semantic_maps")
SCALE   = 0.05   # 米/像素（InteriorGS 默认值；若 occupancy.json 中有其他值请修改）

json_files = sorted(MAP_DIR.glob("2D_Semantic_Map_*_Complete.json"))
print(f"找到 {len(json_files)} 个场景 JSON")

In [ ]:
# ── 核心解析函数 ───────────────────────────────────────────────────────────────

def parse_scene(path: Path, scale: float = SCALE):
    """
    返回 dict 包含：
      scene_id           : str
      total_bbox_area_m2 : 场景坐标边界框面积（所有实例坐标的外包矩形）
      floor_area_m2      : floor 类像素坐标去重后的面积（方案A）
      floor_pixel_sum_m2 : floor 类 area 字段之和 × scale²（方案B）
      n_instances        : 实例总数
      categories         : Counter {label: count}
    """
    with path.open(encoding="utf-8") as f:
        data = json.load(f)

    # scene_id 从文件名中提取，如 2D_Semantic_Map_0001_839920_Complete.json → 0001_839920
    stem = path.stem  # e.g. 2D_Semantic_Map_0001_839920_Complete
    scene_id = stem.replace("2D_Semantic_Map_", "").replace("_Complete", "")

    pixel_area = scale ** 2  # m² per pixel-cell

    all_coords = []   # (y, x) floats for bounding box
    floor_coords = set()  # unique (y_str, x_str) for floor pixels
    floor_pixel_sum = 0
    categories = defaultdict(int)

    for inst in data:
        label = inst.get("category_label", "unknown")
        categories[label] += 1

        coords = inst.get("mask_coords_m", [])
        for c in coords:
            y, x = float(c[0]), float(c[1])
            all_coords.append((y, x))

        if label == "floor":
            for c in coords:
                floor_coords.add((c[0], c[1]))  # deduplicate by string key
            floor_pixel_sum += inst.get("area", 0)

    # 方案A：场景外包矩形
    if all_coords:
        ys = [c[0] for c in all_coords]
        xs = [c[1] for c in all_coords]
        total_bbox_area = (max(ys) - min(ys)) * (max(xs) - min(xs))
    else:
        total_bbox_area = 0.0

    # 方案A：floor 去重像素数 × pixel_area
    floor_area_m2 = len(floor_coords) * pixel_area

    # 方案B：floor area 字段求和 × pixel_area
    floor_pixel_sum_m2 = floor_pixel_sum * pixel_area

    return {
        "scene_id": scene_id,
        "total_bbox_area_m2": round(total_bbox_area, 2),
        "floor_area_m2": round(floor_area_m2, 2),          # 方案A（推荐）
        "floor_pixel_sum_m2": round(floor_pixel_sum_m2, 2), # 方案B
        "n_instances": len(data),
        "categories": dict(categories),
    }

print("函数定义完毕")

In [ ]:
# ── 批量解析所有场景 ──────────────────────────────────────────────────────────
records = []
for p in json_files:
    try:
        rec = parse_scene(p)
        records.append(rec)
    except Exception as e:
        print(f"[ERROR] {p.name}: {e}")

df = pd.DataFrame([
    {k: v for k, v in r.items() if k != "categories"}
    for r in records
])

print(f"成功解析 {len(df)} 个场景")
df.describe()

In [ ]:
# ── 打印统计摘要 ──────────────────────────────────────────────────────────────
for col, label in [("floor_area_m2", "地板面积（方案A，去重坐标）"),
                   ("floor_pixel_sum_m2", "地板面积（方案B，area字段求和）"),
                   ("total_bbox_area_m2", "场景外包矩形总面积")]:
    s = df[col]
    print(f"\n【{label}】")
    print(f"  最小值: {s.min():.1f} m²  最大值: {s.max():.1f} m²")
    print(f"  均值:   {s.mean():.1f} m²  中位数: {s.median():.1f} m²  标准差: {s.std():.1f} m²")
    p25, p75 = s.quantile(0.25), s.quantile(0.75)
    print(f"  Q1:     {p25:.1f} m²  Q3:     {p75:.1f} m²")

In [ ]:
# ── 可视化：面积分布 ──────────────────────────────────────────────────────────
area_col = "floor_area_m2"   # 使用方案A
areas = df[area_col].values

# 面积分级
bins   = [0, 20, 50, 100, 200, 500, np.inf]
labels = ["<20", "20-50", "50-100", "100-200", "200-500", ">500"]
df["area_bin"] = pd.cut(areas, bins=bins, labels=labels)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("SAGE-3D 场景地板面积分布", fontsize=15, fontweight="bold", y=1.02)

# ─ 左：直方图 ─
ax = axes[0]
ax.hist(areas, bins=40, color="#4C72B0", edgecolor="white", linewidth=0.4)
ax.axvline(np.mean(areas),   color="tomato", lw=1.8, linestyle="--", label=f"均值 {np.mean(areas):.0f} m²")
ax.axvline(np.median(areas), color="gold",   lw=1.8, linestyle="--", label=f"中位数 {np.median(areas):.0f} m²")
ax.set_xlabel("地板面积 (m²)")
ax.set_ylabel("场景数量")
ax.set_title("面积直方图")
ax.legend(fontsize=9)

# ─ 中：箱线图 ─
ax2 = axes[1]
ax2.boxplot(areas, vert=True, patch_artist=True,
            boxprops=dict(facecolor="#4C72B0", alpha=0.6),
            medianprops=dict(color="gold", lw=2),
            flierprops=dict(marker="o", markersize=3, color="#4C72B0", alpha=0.4))
ax2.set_ylabel("地板面积 (m²)")
ax2.set_title("箱线图")
ax2.set_xticks([])

# ─ 右：分级柱状图 ─
ax3 = axes[2]
bin_counts = df["area_bin"].value_counts().reindex(labels).fillna(0).astype(int)
colors = ["#5b9bd5", "#70ad47", "#ffc000", "#ff7c2a", "#c00000", "#7030a0"]
bars = ax3.bar(labels, bin_counts.values, color=colors, edgecolor="white", linewidth=0.5)
for bar, cnt in zip(bars, bin_counts.values):
    if cnt > 0:
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(cnt), ha="center", va="bottom", fontsize=10, fontweight="bold")
ax3.set_xlabel("面积区间 (m²)")
ax3.set_ylabel("场景数量")
ax3.set_title("按面积分级")

plt.tight_layout()
plt.savefig("scene_area_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("图表已保存至 scene_area_distribution.png")

In [ ]:
# ── 行人数量推荐 ──────────────────────────────────────────────────────────────
#
# 参考社交密度标准（人/m²）：
#   低密度  0.05  ← 私人住宅、卧室
#   中密度  0.15  ← 普通客厅、餐厅
#   高密度  0.30  ← 繁忙公共区域、走廊

DENSITY = {"low": 0.05, "medium": 0.15, "high": 0.30}

for level, d in DENSITY.items():
    df[f"ped_{level}"] = (df[area_col] * d).clip(lower=1).round().astype(int)

print("\n行人数量推荐（基于地板面积，方案A）：")
print(df[["scene_id", area_col, "area_bin", "ped_low", "ped_medium", "ped_high"]].to_string(index=False))

In [ ]:
# ── 按面积分级的行人推荐汇总 ─────────────────────────────────────────────────
summary = df.groupby("area_bin", observed=True).agg(
    场景数=(area_col, "count"),
    平均面积_m2=(area_col, "mean"),
    中位面积_m2=(area_col, "median"),
    低密度行人=("ped_low", "mean"),
    中密度行人=("ped_medium", "mean"),
    高密度行人=("ped_high", "mean"),
).round(1)

print("\n各面积区间推荐行人数（均值）：")
print(summary.to_string())

# 可视化推荐行人数
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(summary))
w = 0.25
ax.bar(x - w, summary["低密度行人"], width=w, label="低密度 (0.05/m²)", color="#70ad47")
ax.bar(x,     summary["中密度行人"], width=w, label="中密度 (0.15/m²)", color="#ffc000")
ax.bar(x + w, summary["高密度行人"], width=w, label="高密度 (0.30/m²)", color="#c00000")
ax.set_xticks(x)
ax.set_xticklabels(summary.index)
ax.set_xlabel("面积区间 (m²)")
ax.set_ylabel("推荐行人数")
ax.set_title("各面积区间的推荐行人数量（不同社交密度）")
ax.legend()
plt.tight_layout()
plt.savefig("pedestrian_recommendation.png", dpi=150, bbox_inches="tight")
plt.show()
print("图表已保存至 pedestrian_recommendation.png")

In [ ]:
# ── 导出 CSV ──────────────────────────────────────────────────────────────────
out_cols = ["scene_id", "total_bbox_area_m2", "floor_area_m2",
            "floor_pixel_sum_m2", "n_instances", "area_bin",
            "ped_low", "ped_medium", "ped_high"]
df[out_cols].to_csv("scene_areas.csv", index=False)
print("结果已保存至 scene_areas.csv")